In [0]:
import requests
import pandas as pd
from pyspark.sql import SparkSession
from datetime import datetime

In [0]:
url="https://opensky-network.org/api/states/all"

response=requests.get(url)

data1=response.json()

In [0]:
print(data1)

In [0]:
columns=[
"icao24",
"callsign",
"origin_country",
"time_position",
"last_contact",
"longitude",
"latitude",
"baro_altitude",
"on_ground",
"velocity",
"true_track",
"vertical_rate",
"sensors",
"geo_altitude",
"squawk",
"spi",
"position_source"
]

pdf1=pd.DataFrame(data1["states"],columns=columns)

In [0]:
print(pdf1)

Add Ingestion Metadata

Important for batch tracking.

Add:

ingestion timestamp batch date source

In [0]:
from pyspark.sql.functions import *


raw_df1=(

spark.createDataFrame(pdf1)

.withColumn(
"ingestion_timestamp",
current_timestamp()
)

.withColumn(
"batch_date",
current_date()
)

.withColumn(
"source",
lit("OpenSky API")
)

)

Convert to string

In [0]:
from pyspark.sql.functions import col

spark_df1 = raw_df1.withColumn(
    "sensors",
    col("sensors").cast("string")
)

Create Metadata Table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS opensky.bronze.file_tracking_real
(
file_name STRING,
processed_timestamp TIMESTAMP,
status STRING
)

Generate File Name

Save Raw Data

Write CSV to ADLS

In [0]:
%sql
SHOW STORAGE CREDENTIALS;

In [0]:
%sql
LIST 'abfss://real@uralibootcamp.dfs.core.windows.net/'

In [0]:
from datetime import datetime

file_name = (
    "flight_snapshot_"
    + datetime.now().strftime("%Y%m%d")
    + ".csv"
)

raw_path = "abfss://real@uralibootcamp.dfs.core.windows.net/raw/Flights/"

output_path = raw_path + file_name


spark_df1.write \
.mode("append") \
.option("header","true") \
.csv(output_path)